# Pandas data analysis

Pandas organizes tabular data in `DataFrame` objects. This notebook introduces loading, inspecting, cleaning, filtering, and combining data using the local diabetes dataset.

## Setup and a small DataFrame

`pandas` is conventionally imported as `pd`.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/diabetes.csv")

In [2]:
people = pd.DataFrame({
    "name": ["Ahmed", "Ali", "Mona"],
    "age": [20, 22, 19],
    "country": ["Egypt", "Iraq", "Sudan"],
})
people

,name,age,country
0,Ahmed,20,Egypt
1,Ali,22,Iraq
2,Mona,19,Sudan


## Load and inspect data

The dataset path is relative to this notebook directory. Start with a small preview and structural checks before changing the data.

In [3]:
diabetes = pd.read_csv(DATA_PATH)

diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35.0,0,33.6,0.627,50,1
1,1,85,66,29.0,0,26.6,0.351,31,0
2,8,183,64,0.0,0,23.3,0.672,32,1
3,1,89,66,23.0,94,28.1,0.167,21,0
4,0,137,40,35.0,168,43.1,2.288,33,1


In [4]:
print("shape:", diabetes.shape)
print("columns:", diabetes.columns.tolist())
diabetes.info()

shape: (772, 9)
columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
<class 'pandas.DataFrame'>
RangeIndex: 772 entries, 0 to 771
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               772 non-null    int64  
 1   Glucose                   772 non-null    int64  
 2   BloodPressure             772 non-null    int64  
 3   SkinThickness             770 non-null    float64
 4   Insulin                   772 non-null    int64  
 5   BMI                       765 non-null    float64
 6   DiabetesPedigreeFunction  772 non-null    float64
 7   Age                       772 non-null    int64  
 8   Outcome                   772 non-null    int64  
dtypes: float64(3), int64(6)
memory usage: 54.4 KB


In [5]:
diabetes.describe().round(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,772.00,772.00,772.00,770.00,772.00,765.00,772.00,772.00,772.00
mean,3.83,120.84,69.08,20.56,79.39,32.03,0.47,33.25,0.35
std,3.37,31.92,19.31,15.94,115.09,7.80,0.33,11.76,0.48
min,0.00,0.00,0.00,0.00,0.00,0.00,0.08,21.00,0.00
25%,1.00,99.00,62.00,0.00,0.00,27.40,0.24,24.00,0.00
50%,3.00,117.00,72.00,23.00,24.00,32.00,0.37,29.00,0.00
75%,6.00,140.00,80.00,32.00,126.25,36.50,0.62,41.00,1.00
max,17.00,199.00,122.00,99.00,846.00,67.10,2.42,81.00,1.00


## Check data quality

Missing values and duplicate rows can affect analysis. The following cells report them without modifying the source DataFrame.

In [6]:
print("duplicate rows:", diabetes.duplicated().sum())
print("missing values:\n", diabetes.isna().sum())

duplicate rows: 4
missing values:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               2
Insulin                     0
BMI                         7
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


## Clean a working copy

Create a copy before cleaning. Here, missing BMI values are replaced with the column median; rows missing `SkinThickness` are removed for this simple example.

In [7]:
cleaned = diabetes.drop_duplicates().copy()
cleaned["BMI"] = cleaned["BMI"].fillna(cleaned["BMI"].median())
cleaned = cleaned.dropna(subset=["SkinThickness"])

print("shape after cleaning:", cleaned.shape)
print("remaining missing values:\n", cleaned.isna().sum())

shape after cleaning: (766, 9)
remaining missing values:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


## Select, filter, and sort

Use square brackets to select columns or filter rows with a boolean condition.

In [8]:
high_glucose = cleaned.loc[cleaned["Glucose"] > 190, ["Glucose", "BMI", "Outcome"]]
high_glucose.head()

,Glucose,BMI,Outcome
8,197,30.5,1
22,196,39.8,1
185,194,35.9,1
206,196,37.5,1
228,197,36.7,0


In [9]:
cleaned.sort_values("Glucose", ascending=False).head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
661,1,199,76,43.0,0,42.9,1.394,22,1
561,0,198,66,32.0,274,41.3,0.502,28,1
8,2,197,70,45.0,543,30.5,0.158,53,1
228,4,197,70,39.0,744,36.7,2.329,31,0
579,2,197,70,99.0,0,34.7,0.575,62,1


In [10]:
outcome_counts = cleaned["Outcome"].value_counts()
print(outcome_counts)
print("number of outcome categories:", cleaned["Outcome"].nunique())

Outcome
0    498
1    268
Name: count, dtype: int64
number of outcome categories: 2


## Combine DataFrames

`concat` stacks tables by rows by default. With `axis=1`, it places columns side by side; matching indexes are important in that case.

In [11]:
sample_a = cleaned.head(3)
sample_b = cleaned.tail(3)
stacked = pd.concat([sample_a, sample_b], ignore_index=True)

stacked[["Glucose", "BMI", "Outcome"]]

,Glucose,BMI,Outcome
0,148,33.6,1
1,85,26.6,0
2,183,23.3,1
3,121,26.2,0
4,126,30.1,1
5,93,30.4,0


## Key takeaways

- Inspect shape, types, missing values, and duplicates before analysis.
- Keep raw data unchanged and clean a separate working copy.
- Use `.loc` for readable row-and-column selection.